In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from scipy.signal import stft

In [2]:
CACHE_FILE = "ptbxl_preprocessed_cache.npz"
if not os.path.exists(CACHE_FILE):
    raise FileNotFoundError(
        f"Cache file '{CACHE_FILE}' not found. Please run pre_processing.ipynb first!"
    )

print(f"Loading cached dataset from {CACHE_FILE}...")
cached = np.load(CACHE_FILE, allow_pickle=True)

X_train = cached["X_train"]          # Raw 100 Hz 3-lead signals: (N_train, 1000, 3)
y_train = cached["y_train"]          # Binarized 5 superclasses: (N_train, 5)
X_test = cached["X_test"]            # Raw 100 Hz 3-lead signals: (N_test, 1000, 3)
y_test = cached["y_test"]            # Binarized 5 superclasses: (N_test, 5)
class_names = cached["class_names"]  # ['CD', 'HYP', 'MI', 'NORM', 'STTC']

X_feat_train = cached["X_feat_train"]  # Morphological & HRV features: (N_train, D)
X_feat_test = cached["X_feat_test"]    # Morphological & HRV features: (N_test, D)

print(f"Loaded successfully in seconds!")
print(f"Signals   -> Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features  -> Train: {X_feat_train.shape} | Test: {X_feat_test.shape}")
print(f"Targets   -> Classes ({len(class_names)}): {class_names.tolist()}")

Loading cached dataset from ptbxl_preprocessed_cache.npz...
Loaded successfully in seconds!
Signals   -> Train: (19230, 1000, 3) | Test: (2158, 1000, 3)
Features  -> Train: (19230, 36) | Test: (2158, 36)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']


In [4]:
def batch_compute_stft(X_signals, fs=100, nperseg=32, noverlap=16):
    """
    Converts (N, 1000, 3) raw time series into (N, 16, 32, 3) STFT spectrogram grids.
    Matches dimensions achievable with hardware-accelerated esp-dsp routines.
    """
    N, _, channels = X_signals.shape
    specs = np.zeros((N, 16, 32, channels), dtype=np.float32)
    
    for i in range(N):
        for c in range(channels):
            _, _, Zxx = stft(X_signals[i, :, c], fs=fs, nperseg=nperseg, noverlap=noverlap)
            # Diagnostic frequency slice (< 25 Hz) and crop first 32 time windows
            mag = np.abs(Zxx[:16, :32])
            log_spec = np.log1p(mag)
            ptp = np.ptp(log_spec)
            specs[i, :, :, c] = (log_spec - np.min(log_spec)) / (ptp + 1e-8) if ptp > 0 else log_spec
            
    return specs

print("Computing STFT spectrogram grids for Train and Test sets...")
X_spec_train = batch_compute_stft(X_train)
X_spec_test = batch_compute_stft(X_test)
print(f"Spectrogram Tensors: Train {X_spec_train.shape} | Test {X_spec_test.shape}")

def build_time_frequency_2d_cnn(input_shape=(16, 32, 3), num_classes=5):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.SeparableConv2D(16, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.SeparableConv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.SeparableConv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="sigmoid")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="TimeFreq_2D_CNN")

cnn_2d = build_time_frequency_2d_cnn(input_shape=X_spec_train.shape[1:], num_classes=len(class_names))
cnn_2d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="macro_auc")]
)

print("\nTraining Time-Frequency 2D-CNN...")
cb = [
    callbacks.EarlyStopping(monitor="val_macro_auc", mode="max", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]
cnn_2d.fit(
    X_spec_train, y_train,
    validation_split=0.1,
    epochs=25,
    batch_size=64,
    callbacks=cb,
    verbose=1
)

cnn_2d_probs = cnn_2d.predict(X_spec_test, batch_size=128)

Computing STFT spectrogram grids for Train and Test sets...
Spectrogram Tensors: Train (19230, 16, 32, 3) | Test (2158, 16, 32, 3)

Training Time-Frequency 2D-CNN...
Epoch 1/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.4876 - macro_auc: 0.7101 - val_loss: 0.5658 - val_macro_auc: 0.6147 - learning_rate: 0.0010
Epoch 2/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.4219 - macro_auc: 0.7969 - val_loss: 0.5009 - val_macro_auc: 0.7823 - learning_rate: 0.0010
Epoch 3/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.4073 - macro_auc: 0.8130 - val_loss: 0.4280 - val_macro_auc: 0.8149 - learning_rate: 0.0010
Epoch 4/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.3995 - macro_auc: 0.8215 - val_loss: 0.4229 - val_macro_auc: 0.8212 - learning_rate: 0.0010
Epoch 5/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.3942 - macro_auc: 0.8264 - val_loss: 0.4184 - val_macro_auc: 0.8249 - learning_rate: 0.0010
Epoch 6/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.3

In [5]:
all_predictions = {}
all_predictions["Time-Freq 2D-CNN (STFT)"] = cnn_2d_probs
print(f"Time-Frequency 2D-CNN Macro AUC: {roc_auc_score(y_test, cnn_2d_probs, average='macro'):.4f}")

Time-Frequency 2D-CNN Macro AUC: 0.8284


In [7]:
import numpy as np

# For 2D-CNN (Notebook 4)
np.savez_compressed("preds_2dcnn.npz", cnn2d=cnn_2d_probs)
cnn_2d.save("timefreq_2d_cnn.keras")